# 03 — Fair comparison: accuracy, latency, params

Cache-first view of the locked roster (`toxic-bert`, `tfidf-svm`, `distilgpt2-d1`,
`distilgpt2-d1-ft`, `distilbert-student`). Prefers `data/processed/model_comparison.json`
(+ optional agreement). Set `LIVE_RERUN = True` only for a small optional re-score.

CLI equivalent: `python scripts/run_model_comparison.py`


In [ ]:
%matplotlib inline
import json, sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Walk up until DeepProjectHate/src/paths.py is found (fixes wrong-cwd PathLike/None errors).
_src = None
for _p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    _cand = _p / "src"
    if (_cand / "paths.py").is_file():
        _src = str(_cand)
        break
if _src is None:
    raise FileNotFoundError(f"Cannot find DeepProjectHate/src/paths.py from cwd={Path.cwd()}")
if _src not in sys.path:
    sys.path.insert(0, _src)

for _mod in ("paths", "roster", "model_registry", "baselines", "splits", "device"):
    sys.modules.pop(_mod, None)

from paths import PROCESSED_DIR, PROJECT_ROOT, SEED
from roster import FAIR_ROSTER, print_roster_status

np.random.seed(SEED)
sns.set_theme(style="whitegrid")
print(f"Project root: {PROJECT_ROOT}")
print(f"src: {_src}")
print("Roster:")
READY = print_roster_status(FAIR_ROSTER)

LIVE_RERUN = False  # optional small live pass
LIVE_EVAL_N = 40
SPLIT = "test"


## 1. Load cached comparison


In [ ]:
COMPARISON = PROCESSED_DIR / "model_comparison.json"
AGREEMENT = PROCESSED_DIR / "agreement_results.json"

if not COMPARISON.exists():
    raise FileNotFoundError(
        f"Missing {COMPARISON}. Run:\n"
        "  python scripts/run_model_comparison.py --quick "
        "--models toxic-bert tfidf-svm distilgpt2-d1"
    )

payload = json.loads(COMPARISON.read_text(encoding="utf-8"))
rows = []
for key in FAIR_ROSTER:
    block = payload.get("models", {}).get(key)
    if block is None:
        print(f"  [CACHE MISS] {key} — not in {COMPARISON.name}")
        continue
    split = block.get("splits", {}).get(SPLIT, {})
    rows.append({
        "model": key,
        "family": block.get("family"),
        "n_params": block.get("n_params"),
        "f1": split.get("f1"),
        "accuracy": split.get("accuracy"),
        "roc_auc": split.get("roc_auc"),
        "pr_auc": split.get("pr_auc"),
        "ms_per_comment": split.get("ms_per_comment"),
        "precision": split.get("precision"),
        "recall": split.get("recall"),
    })

metrics = pd.DataFrame(rows)
display(metrics.round(3))
print(f"Loaded {COMPARISON} (split={SPLIT}, quick_mode={payload.get('quick_mode')})")


## 2. Plots (F1 / latency / params)


In [ ]:
if metrics.empty:
    raise RuntimeError("No roster models found in cache — run model comparison first.")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.barplot(data=metrics, x="model", y="f1", ax=axes[0], color="steelblue")
axes[0].set_title(f"F1 ({SPLIT})")
axes[0].tick_params(axis="x", rotation=30)
axes[0].set_ylim(0, 1)

sns.barplot(data=metrics, x="model", y="ms_per_comment", ax=axes[1], color="darkorange")
axes[1].set_title("Latency (ms/comment)")
axes[1].tick_params(axis="x", rotation=30)

sns.barplot(data=metrics, x="model", y="n_params", ax=axes[2], color="seagreen")
axes[2].set_title("Parameters")
axes[2].tick_params(axis="x", rotation=30)
axes[2].ticklabel_format(style="sci", axis="y", scilimits=(6, 6))
plt.tight_layout()
plt.show()


## 3. Optional agreement cache


In [ ]:
if AGREEMENT.exists():
    ag = json.loads(AGREEMENT.read_text(encoding="utf-8"))
    print(f"Loaded {AGREEMENT}")
    print(json.dumps(ag if not isinstance(ag, dict) else {k: ag[k] for k in list(ag)[:8]}, indent=2)[:2000])
else:
    print(f"No agreement cache at {AGREEMENT} (optional). Run: python scripts/run_agreement.py --n 150")


## 4. Optional live re-run (small)


In [ ]:
if not LIVE_RERUN:
    print("LIVE_RERUN=False — skipping live scoring (cache-first).")
else:
    from model_registry import ALL_SPECS, load_scorer
    from splits import get_split_frame, sample_split_frame

    frame = sample_split_frame(SPLIT, n=LIVE_EVAL_N, seed=SEED)
    texts = frame["comment_text"].astype(str).tolist()
    y = frame["toxic"].astype(int).values
    live_rows = []
    for key in READY:
        spec = ALL_SPECS[key]
        try:
            scorer = load_scorer(spec)
            if key == "tfidf-svm":
                train = get_split_frame("train")
                scorer.fit(train["comment_text"].tolist(), train["toxic"].values)
            t0 = time.perf_counter()
            p = scorer.predict_toxic_proba(texts)
            ms = 1000.0 * (time.perf_counter() - t0) / max(len(texts), 1)
            pred = (np.asarray(p) >= 0.5).astype(int)
            live_rows.append({
                "model": key,
                "accuracy": float((pred == y).mean()),
                "ms_per_comment": ms,
                "n_params": getattr(scorer, "n_params", None),
            })
        except Exception as exc:
            print(f"  [SKIP live] {key}: {exc}")
    display(pd.DataFrame(live_rows).round(3))
